 ### CAPSTONE PROJECT 3: K beauty AI assistant
by Irina Kim


In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 1. LOAD & INITIALIZE DATA
# ==========================================
# Load the wrangled data generated in the EDA step
df = pd.read_csv('wrangled_kbeauty_data.csv')

# Knowledge Base: Mapping Concerns to Hero Ingredients
# This acts as our "Expert System" logic
concern_map = {
    'acne': 'salicylic bha tea tree benzoyl succinic',
    'hyperpigmentation': 'vitamin c niacinamide arbutin tranexamic licorice kojic',
    'sun stains': 'vitamin c niacinamide arbutin glycolic',
    'wrinkles': 'peptide retinol bakuchiol collagen ginseng adenosine',
    'sensitive': 'centella cica mugwort ceramide panthenol squalane',
    'dark skin': 'niacinamide tranexamic arbutin azelaic'
}

# ==========================================
# 2. NLP ENGINE: SEMANTIC SEARCH (TF-IDF)
# ==========================================
# We vectorize the ingredients so we can compare "vague user text" to "product formulas"
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['cleaned_ingredients'].fillna(''))

def find_clinical_matches(user_query, top_n=10):
    """Uses TF-IDF to find products whose ingredients match the user's skin query."""
    query_vec = tfidf.transform([user_query])
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()
    indices = similarity.argsort()[-top_n:][::-1]
    return df.iloc[indices].copy()

# ==========================================
# 3. THE ARCHITECT: ROUTINE & BUDGET LOGIC
# ==========================================
def k_beauty_spa_assistant(query, target_tier='Medium/Optimal'):
    """
    Builds a full 4-step routine based on a specific concern and budget tier.
    Tiers: 'Costco/Value', 'Budget', 'Medium/Optimal', 'Luxury/Clinical'
    """
    # 1. Get products that clinically match the concern
    candidates = find_clinical_matches(query, top_n=50)
    
    # 2. Filter for the requested tier (or nearest tier if empty)
    tier_matches = candidates[candidates['tier'] == target_tier]
    if tier_matches.empty:
        tier_matches = candidates # Fallback to any tier if the specific one is empty
        
    # 3. Categorize into Steps
    routine = {}
    steps = {
        "Step 1: Cleanse": "Cleanser",
        "Step 2: Treat (Serum)": "Treatments",
        "Step 3: Seal (Moisturize)": "Moisturizers",
        "Step 4: Protect (SPF)": "Sunscreen"
    }
    
    total_cost = 0
    
    for label, category in steps.items():
        # Filter by category string
        step_df = tier_matches[tier_matches['secondary_category'].str.contains(category, na=False, case=False)]
        
        if not step_df.empty:
            # Pick the best-rated product in that category
            rec = step_df.sort_values(by='rating', ascending=False).iloc[0]
            routine[label] = {
                "Product": rec['product_name'],
                "Brand": rec['brand_name'],
                "Price": f"${rec['price_usd']}",
                "Why": "Clinically matched to your skin profile"
            }
            total_cost += rec['price_usd']
        else:
            routine[label] = {"Product": "Explore Spa Professional line", "Brand": "In-Shop", "Price": "N/A"}

    # 4. Spa Procedure Recommendation Logic
    procedure = "General maintenance."
    if any(word in query.lower() for word in ['wrinkle', 'age', 'lines']):
        procedure = "Neurotoxins (Botox) + In-spa Peptide Infusion."
    elif any(word in query.lower() for word in ['stain', 'hyperpigmentation', 'dark']):
        procedure = "Chemical Peel (safe for melanin) or Microneedling with Brightening Serum."

    return routine, total_cost, procedure

# ==========================================
# 4. EXAMPLE USAGE & OUTPUT
# ==========================================
user_need = "I have sun stains and I'm looking for a Costco budget routine"
routine, cost, proc = k_beauty_spa_assistant(user_need, target_tier='Costco/Value')

print(f"REPORT FOR: {user_need.upper()}")
print(f"Target Budget Tier: Costco/Value")
print("-" * 30)
for step, details in routine.items():
    print(f"{step}: {details['Product']} by {details['Brand']} ({details['Price']})")

print("-" * 30)
print(f"ESTIMATED TOTAL: ${cost:.2f}")
print(f"PROFESSIONAL ADVICE: {proc}")



REPORT FOR: I HAVE SUN STAINS AND I'M LOOKING FOR A COSTCO BUDGET ROUTINE
Target Budget Tier: Costco/Value
------------------------------
Step 1: Cleanse: Cactus Water Cleansing Lactic Acid Toner by Freck Beauty ($32.0)
Step 2: Treat (Serum): Tea Elixir Niacinamide & Hyaluronic Acid Anti-Aging Serum by fresh ($80.0)
Step 3: Seal (Moisturize): Black Tea Anti-Aging Moisturizer with Retinol-Alternative BT Matrix by fresh ($95.0)
Step 4: Protect (SPF): Mineral Sunscreen Zinc Oxide Broad Spectrum SPF 30 by First Aid Beauty ($28.0)
------------------------------
ESTIMATED TOTAL: $235.00
PROFESSIONAL ADVICE: Chemical Peel (safe for melanin) or Microneedling with Brightening Serum.
